In [ ]:
import pandas as pd

df = pd.read_csv('data_diabetes/diabetes_prediction_dataset.csv')
print(df.shape)
df.head()

(100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [60]:
target_col = 'diabetes'

In [61]:
def clean_data(df):
    print(f"Initial shape: {df.shape}")
    
    # Duplicates
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        df = df.drop_duplicates()
        print(f"Removed {dup_count} duplicate rows.")
    
    # Drop fully-empty columns
    empty_cols = df.columns[df.isnull().all()]
    if len(empty_cols) > 0:
        df = df.drop(columns=empty_cols)
        print(f"Dropped fully-empty columns: {list(empty_cols)}")
    
    # Drop id-like columns (common naming patterns)
    id_cols = [col for col in df.columns if col.lower() in ['id', 'patient_id', 'index']]
    if id_cols:
        df = df.drop(columns=id_cols)
        print(f"Dropped ID columns: {id_cols}")
    
    # Report remaining missing values
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    if len(cols_with_nulls) > 0:
        print(f"Columns with remaining missing values:\n{cols_with_nulls}")
    else:
        print("No missing values remaining.")
    
    print(f"Final shape: {df.shape}")
    return df

In [62]:
df=clean_data(df)

Initial shape: (100000, 9)
Removed 3854 duplicate rows.
No missing values remaining.
Final shape: (96146, 9)


In [63]:
df.info()
print(df[target_col].isnull().sum())
print(df[target_col].unique())

<class 'pandas.DataFrame'>
Index: 96146 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               96146 non-null  str    
 1   age                  96146 non-null  float64
 2   hypertension         96146 non-null  int64  
 3   heart_disease        96146 non-null  int64  
 4   smoking_history      96146 non-null  str    
 5   bmi                  96146 non-null  float64
 6   HbA1c_level          96146 non-null  float64
 7   blood_glucose_level  96146 non-null  int64  
 8   diabetes             96146 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 7.3 MB
0
[0 1]


In [64]:
# Cell: define target column (reusable variable, not hardcoded everywhere)
target_col = 'diabetes'

# Cell: define the function (if not already defined)
def encode_categoricals(df, target_col):
    feature_cols = [col for col in df.columns if col != target_col]
    categorical_feature_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
    
    if categorical_feature_cols:
        print(f"One-hot encoding: {categorical_feature_cols}")
        df = pd.get_dummies(df, columns=categorical_feature_cols)
    else:
        print("No categorical features to encode.")
    
    return df

# Cell: call it
df = encode_categoricals(df, target_col)

One-hot encoding: ['gender', 'smoking_history']


/tmp/ipykernel_2794/1909822800.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_feature_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()


In [65]:
def check_imbalance(df, target_col):
    counts = df[target_col].value_counts()
    percentages = df[target_col].value_counts(normalize=True) * 100
    
    print("Class counts:")
    print(counts)
    print("\nClass percentages:")
    print(percentages.round(2))
    
    minority_pct = percentages.min()
    if minority_pct < 10:
        print(f"\n⚠️ Severe imbalance detected (minority class: {minority_pct:.1f}%). Consider resampling or class_weight='balanced'.")
    elif minority_pct < 30:
        print(f"\n⚠️ Mild-to-moderate imbalance (minority class: {minority_pct:.1f}%). Use class_weight='balanced' and check precision/recall, not just accuracy.")
    else:
        print(f"\n Classes are reasonably balanced (minority class: {minority_pct:.1f}%).")
    
    return counts

check_imbalance(df, target_col)

Class counts:
diabetes
0    87664
1     8482
Name: count, dtype: int64

Class percentages:
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64

⚠️ Severe imbalance detected (minority class: 8.8%). Consider resampling or class_weight='balanced'.


diabetes
0    87664
1     8482
Name: count, dtype: int64

In [66]:
if df[target_col].dtype == 'object':
    df[target_col] = df[target_col].map({'M': 1, 'B': 0})
    print("Target encoded from text to 0/1")
else:
    print("Target already numeric, no encoding needed")
df[target_col].value_counts()

Target already numeric, no encoding needed


diabetes
0    87664
1     8482
Name: count, dtype: int64

In [67]:
def check_outliers(df, target_col):
    feature_cols = [col for col in df.columns if col != target_col]
    # Only check numeric columns, excluding booleans (from one-hot encoding)
    numeric_cols = df[feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    outlier_summary = {}
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        if len(outliers) > 0:
            outlier_summary[col] = len(outliers)
    
    print(f"Columns with outliers (IQR method):")
    for col, count in sorted(outlier_summary.items(), key=lambda x: -x[1]):
        print(f"  {col}: {count} outliers")
    
    return outlier_summary

outlier_summary = check_outliers(df, target_col)

Columns with outliers (IQR method):
  hypertension: 7461 outliers
  bmi: 5354 outliers
  heart_disease: 3923 outliers
  blood_glucose_level: 2031 outliers
  HbA1c_level: 1312 outliers


In [68]:
df['high_hba1c'] = (df['HbA1c_level'] >= 6.5).astype(int)
df['high_glucose'] = (df['blood_glucose_level'] >= 200).astype(int)
df['clinical_risk_flag'] = ((df['high_hba1c'] == 1) | (df['high_glucose'] == 1)).astype(int)

print(df[['HbA1c_level', 'blood_glucose_level', 'high_hba1c', 'high_glucose', 'clinical_risk_flag', target_col]].head(10))

   HbA1c_level  blood_glucose_level  high_hba1c  high_glucose  \
0          6.6                  140           1             0   
1          6.6                   80           1             0   
2          5.7                  158           0             0   
3          5.0                  155           0             0   
4          4.8                  155           0             0   
5          6.6                   85           1             0   
6          6.5                  200           1             1   
7          5.7                   85           0             0   
8          4.8                  145           0             0   
9          5.0                  100           0             0   

   clinical_risk_flag  diabetes  
0                   1         0  
1                   1         0  
2                   0         0  
3                   0         0  
4                   0         0  
5                   1         0  
6                   1         1  
7           

In [69]:
#splitting into features
X = df.drop(columns=[target_col])
y = df[target_col]

print(X.shape)
print(y.shape)

(96146, 18)
(96146,)


In [70]:
from sklearn.model_selection import train_test_split
def split_data(X, y, is_classification=True, test_size=0.2):
    if is_classification:
        return train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    else:
        return train_test_split(X, y, test_size=test_size, random_state=42)

In [71]:
X_train, X_test, y_train, y_test = split_data(X, y)

In [72]:
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (76916, 18)
Test shape: (19230, 18)


In [73]:
def choose_scaler(outlier_summary, total_features):
    outlier_ratio = len(outlier_summary) / total_features
    if outlier_ratio > 0.3:
        print(f"Significant outliers detected ({len(outlier_summary)}/{total_features} features) → using RobustScaler")
        from sklearn.preprocessing import RobustScaler
        return RobustScaler()
    else:
        print("Few outliers detected → using StandardScaler")
        from sklearn.preprocessing import StandardScaler
        return StandardScaler()

scaler = choose_scaler(outlier_summary, total_features=X.shape[1])

Few outliers detected → using StandardScaler


In [74]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete")
print(X_train_scaled.shape)
print(X_test_scaled.shape)

Scaling complete
(76916, 18)
(19230, 18)


In [75]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained")

Model trained


In [76]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nFull Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8812
Precision: 0.4190
Recall: 0.8968
F1 Score: 0.5712

Confusion Matrix:
[[15425  2109]
 [  175  1521]]

Full Report:
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     17534
           1       0.42      0.90      0.57      1696

    accuracy                           0.88     19230
   macro avg       0.70      0.89      0.75     19230
weighted avg       0.94      0.88      0.90     19230



In [77]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1: {f1_score(y_test, y_pred_rf):.4f}")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.9574
Precision: 0.7646
Recall: 0.7471
F1: 0.7557
[[17144   390]
 [  429  1267]]


In [78]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)

y_pred_gb = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")

print(f"Precision: {precision_score(y_test, y_pred_gb):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_gb):.4f}")
print(f"F1: {f1_score(y_test, y_pred_gb):.4f}")
print(confusion_matrix(y_test, y_pred_gb))


Accuracy: 0.9717
Precision: 0.9873
Recall: 0.6881
F1: 0.8110
[[17519    15]
 [  529  1167]]


In [79]:
y_proba_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.5, 0.4, 0.3, 0.2, 0.15]:
    y_pred_t = (y_proba_gb >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold {threshold}: Precision={p:.4f}, Recall={r:.4f}, F1={f1:.4f}")

Threshold 0.5: Precision=0.9873, Recall=0.6881, F1=0.8110
Threshold 0.4: Precision=0.9458, Recall=0.7099, F1=0.8110
Threshold 0.3: Precision=0.8620, Recall=0.7441, F1=0.7987
Threshold 0.2: Precision=0.7046, Recall=0.8072, F1=0.7524
Threshold 0.15: Precision=0.6118, Recall=0.8485, F1=0.7110


In [80]:
correlations = df.corr(numeric_only=True)[target_col].sort_values(ascending=False)
print(correlations)

diabetes                       1.000000
blood_glucose_level            0.424336
HbA1c_level                    0.406408
clinical_risk_flag             0.350381
high_glucose                   0.349584
high_hba1c                     0.307110
age                            0.264927
bmi                            0.214932
hypertension                   0.195710
heart_disease                  0.170711
smoking_history_former         0.095492
gender_Male                    0.037883
smoking_history_never          0.023136
smoking_history_ever           0.021915
smoking_history_not current    0.018921
smoking_history_current        0.017037
gender_Other                  -0.004256
gender_Female                 -0.037763
smoking_history_No Info       -0.112576
Name: diabetes, dtype: float64


In [81]:
lr_with_features = LogisticRegression(class_weight='balanced', random_state=42)
lr_with_features.fit(X_train_scaled, y_train)
y_pred_lr_new = lr_with_features.predict(X_test_scaled)
print(f"Precision: {precision_score(y_test, y_pred_lr_new):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr_new):.4f}")
print(f"F1: {f1_score(y_test, y_pred_lr_new):.4f}")

Precision: 0.4190
Recall: 0.8968
F1: 0.5712
